# Player EDA

Per-game player summaries, leaders, variation, shooting, minutes and season trends. Only fields present in the selected statistics table are calculated.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'backend' / 'app').is_dir())
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
import pandas as pd
import plotly.express as px
from backend.app import eda

DATA_DIRS = [ROOT / 'data' / 'processed', ROOT / 'data' / 'raw', ROOT / 'Raw']
player_result = eda.players_analysis(DATA_DIRS)
if player_result['available']:
    print('Rows:', player_result['game_count'], '| players:', player_result['player_count'])
    print('Metrics:', player_result['supported_metrics'])
    print('Unavailable:', player_result['unavailable_analyses'])
    display(pd.DataFrame(player_result['leaders'].get('points', [])))
else:
    print(player_result['reason'])

In [ ]:
chart = player_result.get('chart_data', {})
if chart.get('players'):
    px.bar(chart, x='players', y='points', title='Top players by average points per game').show()
    px.scatter(chart, x='rebounds', y='points', hover_name='players',
               title='Player scoring and rebounding comparison').show()
    px.box(y=chart.get('points_boxplot', []), points='outliers',
           labels={'y': 'Average points per game'}, title='Distribution of player scoring averages').show()

In [ ]:
player_relationships = eda.correlations_analysis(DATA_DIRS).get('player_relationships', {})
for name, relationship in player_relationships.items():
    px.scatter(x=relationship['x'], y=relationship['y'],
               labels={'x': relationship['x_label'], 'y': relationship['y_label']},
               title=f"Player performance relationship: {name}").show()
season = pd.DataFrame(player_result.get('season_performance', []))
if not season.empty and 'points' in season:
    candidates = season.groupby('player')['points'].mean().nlargest(8).index
    px.line(season[season['player'].isin(candidates)], x='season', y='points',
            color='player', markers=True, title='Top scorer season trends').show()

In [ ]:
sources = eda._source_files(DATA_DIRS)
player_path = eda._find_source(sources, ('playerstatisticsextended', 'playerstatistics'))
if player_path:
    game_stats = eda._read(player_path)
    points_column = eda._find_metric(game_stats, 'points')
    if points_column:
        px.histogram(game_stats, x=points_column, nbins=50,
                     title='Player game-level points distribution').show()
    else:
        print('Player point distribution unavailable: no points field.')
else:
    print('Player statistics table unavailable.')